In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler


def scaling_features(train_set, val_set, method):
    """ Scales the features of the train and validation sets according to the specified method.
    Args:
        train_set (pd.DataFrame): The training data to be scaled.
        val_set (pd.DataFrame): The validation data to be scaled.
        method (str): The scaling method to use. Options are 'minmax' - between 0 and 1, 'minmax2' - between -1 and 1, 
        'standard', and 'robust'.
    Returns:
        scaled_X_train (np.ndarray): The scaled training data.
        scaled_X_val (np.ndarray): The scaled validation data.
    """

    if method == 'minmax':
        #scale your data using MinMaxScaler[0,1]
        min_max = MinMaxScaler().fit(train_set)
        # Transform your train data by applying the scale obtained in the previous command
        scaled_X_train = min_max.transform(train_set)
        # Transform your validation data by applying the scale obtained in the first command
        scaled_X_val = min_max.transform(val_set)
    elif method == 'minmax2':
        # Create a MinMaxScaler instance that will range between -1 and 1 and fit to your train data
        min_max = MinMaxScaler(feature_range=(-1, 1)).fit(train_set)
        # Transform your train data by applying the scale obtained in the previous command
        scaled_X_train = min_max.transform(train_set)
        # Transform your validation data by applying the scale obtained in the first command
        scaled_X_val = min_max.transform(val_set)
    elif method == 'standard':
        # Create a StandardScaler instance and fit to your train data
        standard = StandardScaler().fit(train_set)
        # Transform your train data by applying the scale obtained in the previous command
        scaled_X_train = standard.transform(train_set)
        # Transform your validation data by applying the scale obtained in the first command
        scaled_X_val = standard.transform(val_set)
    else: 
        robust = RobustScaler().fit(train_set)
        # Transform your train data by applying the scale obtained in the previous command
        scaled_X_train = robust.transform(train_set)
        # Transform your validation data by applying the scale obtained in the first command
        scaled_X_val = robust.transform(val_set)      
    return scaled_X_train, scaled_X_val

In [ ]:

def imputation(data, train, val, test, num_method, cat_method, threshold=5.0, neighbors=5):

    # Create copies of the training and validation datasets
    train_copy = train.copy().reset_index()
    val_copy = val.copy().reset_index()
    test_copy = test.copy().reset_index()

    categorical = ['Brand', 'model', 'transmission', 'fuelType', 'hasDamage', 'is_recent_car', 'mileage_category',
            'is_hybrid_or_electric', 'is_automatic', 'paintQuality_category', 'has_damage_or_low_paint', 'is_first_owner']
    numerical = data.drop(categorical, axis=1).columns.tolist()

    missing_percentages_df = missing_values_table(data)
    low_missing_values = missing_percentages_df[missing_percentages_df['Missing_Percent'] <= threshold]['Feature'].tolist()
    high_missing_values = missing_percentages_df[missing_percentages_df['Missing_Percent'] > threshold]['Feature'].tolist()

    # -------------  LOW MISSING  -------------
    # Numerical Variables - Median Imputation
    num_low = [n for n in low_missing_values if n in numerical]
    if num_low:
        # calculate median from training set
        median_value = train_copy[num_low].median()

        # fill missing values in train, val, and test sets with train median
        for df in [train_copy, val_copy, test_copy]:
            df[num_low].fillna(median_value, inplace=True)

    # Categorical Variables - Mode Imputation
    cat_low = [c for c in low_missing_values if c in categorical]
    if cat_low:
        # calculate mode from training set
        mode_value = train_copy[cat_low].mode().iloc[0]
        # fill missing values in train, val, and test sets with train mode
        for df in [train_copy, val_copy, test_copy]:
            df[cat_low].fillna(mode_value, inplace=True)
    
    # -------------  HIGH MISSING  -------------
    num_high = [n for n in high_missing_values if n in numerical]
    cat_high = [c for c in high_missing_values if c in categorical]

    # Numerical Variables
    if num_high:
        # KNN Imputation
        if method == 'KNN':
            # Fit the KNNImputer on the training set
            knn_imputer = KNNImputer(n_neighbors=neighbors)
            knn_imputer.fit(train_copy[num_high])

            # Transform training, validation, and test sets
            for df in [train_copy, val_copy, test_copy]:
                df[num_high] = pd.DataFrame(knn_imputer.transform(df[num_high]),
                                            columns=num_high,
                                            index=df.index)
                   
        # MICE Imputation
        elif method == 'Iterative':
            # Fit the IterativeImputer on the training set
            iterative_imputer = IterativeImputer(random_state=40111)
            iterative_imputer.fit(train_copy[num_high])

            # Transform training, validation, and test sets
            for df in [train_copy, val_copy, test_copy]:
                df[num_high] = pd.DataFrame(iterative_imputer.transform(df[num_high]),
                                            columns=num_high,
                                            index=df.index)
        # Random Forest Imputation for Numerical Columns
        elif method == 'RF'
                not_missing = train_copy[train_copy[col].notna()]
                missing = train_copy[train_copy[col].isna()]
                if not not_missing.empty:
                    rf = RandomForestRegressor(n_estimators=200, random_state=40111, n_jobs=-1)
                    # Fit RandomForestRegressor with training data without missing values
                    rf.fit(not_missing.drop(columns=[col]), not_missing[col])
                    
                    # Fill train missing
                    if not missing.empty:
                        train_copy.loc[missing.index, col] = rf.predict(missing.drop(columns=[col]))
                    
                    # Fill val missing
                    mask = val_copy[col].isna()
                    if mask.sum() > 0:
                        val_copy.loc[mask, col] = rf.predict(val_copy.loc[mask].drop(columns=[col]))
                    
                    # Fill test missing
                    mask = test_copy[col].isna()
                    if mask.sum() > 0:
                        test_copy.loc[mask, col] = rf.predict(test_copy.loc[mask].drop(columns=[col]))

    if cat_high:
        # Mode Imputation
        if method == 'Mode':
            mode_value = train_copy[cat_high].mode().iloc[0]
            # fill missing values in train, val, and test sets with train mode
            for df in [train_copy, val_copy, test_copy]:
                df[cat_high].fillna(mode_value, inplace=True)

        # Random Forest Imputation for Categorical Columns
        elif method == 'RF':
            not_missing = train_copy[train_copy[col].notna()]
            missing = train_copy[train_copy[col].isna()]

            if not not_missing.empty:
                # Define features (all columns except target)
                feature_cols = [c for c in train_copy.columns if c != col]

                # Train RandomForestClassifier
                clf = RandomForestClassifier(n_estimators=200, random_state=40111, n_jobs=-1)
                clf.fit(not_missing[feature_cols], not_missing[col])

                # Fill train missing
                if not missing.empty:
                    train_copy.loc[missing.index, col] = clf.predict(missing[feature_cols])

                # Fill val missing
                mask = val_copy[col].isna()
                if mask.sum() > 0:
                    val_copy.loc[mask, col] = clf.predict(val_copy.loc[mask, feature_cols])

                # Fill test missing
                mask = test_copy[col].isna()
                if mask.sum() > 0:
                    test_copy.loc[mask, col] = clf.predict(test_copy.loc[mask, feature_cols])



    return train_set_copy, val_set_copy, test_set_copy